# End-to-end capstone run

Requires Redis running and the worker started in a separate terminal.

In [ ]:
import httpx, pathlib, time
pdf = pathlib.Path('sample.pdf')
assert pdf.exists(), 'put sample.pdf next to this notebook'
r = httpx.post('http://localhost:8000/upload', files={'file': (pdf.name, pdf.read_bytes(), 'application/pdf')})
fid = r.json()['file_id']
job = httpx.post(f'http://localhost:8000/summarize/{fid}').json()
print(job)

In [ ]:
job_id = job['job_id']
for _ in range(40):
    s = httpx.get(f'http://localhost:8000/jobs/{job_id}').json()
    print(s['status'], s.get('chunks_done'), '/', s.get('chunks_total'))
    if s['status'] in ('done','failed'):
        print('SUMMARY:', s.get('summary')[:500] if s.get('summary') else '')
        break
    time.sleep(1)